# 1. Setup

In [21]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [22]:
data = pd.read_csv("../data/raw/online_retail_II.csv")

In [35]:
def clean_data(data):
    df = data.dropna()
    df = df.drop_duplicates()
    df = df[df["Price"] > 0]
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    return df

In [36]:
data = clean_data(data)

# 2. Churn Definition

In [125]:
CUTOFF = pd.Timestamp("2011-09-09")

A customer is labeled as **churned (1)** if they made at least one purchase before the cutoff date (2011-09-09) but made **no purchase in the following 90 days** (2011-09-09 to 2011-12-09). Otherwise they are labeled **active (0)**.

- **Features** are built only from data *before* the cutoff.
- **Labels** come only from the 90 days *after* the cutoff.

This prevents data leakage: the model predicts future behavior using only past information.

In [126]:
def build_labels(data,cutoff):
    before_cutoff = data[data["InvoiceDate"] < cutoff]
    after_cut_off = data[data["InvoiceDate"] > cutoff]

    before_customer = before_cutoff["Customer ID"].unique()
    after_customer = after_cut_off["Customer ID"].unique()
    
    label = pd.DataFrame({"Customer ID":before_customer})
    label["returned"] = label["Customer ID"].isin(after_customer).astype(int)
    return label

In [127]:
labels = build_labels(data,CUTOFF)

In [128]:
labels['returned'].value_counts()

returned
0    3013
1    2328
Name: count, dtype: int64

# 3. Feature Engineering

In [129]:
data

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [138]:
def build_features(data, cutoff):
    before_cutoff = data[data["InvoiceDate"] < cutoff]
    # after_cut_off removed: features must not see the future

    first_purchase = before_cutoff.groupby("Customer ID")["InvoiceDate"].min()
    last_purchase = before_cutoff.groupby("Customer ID")["InvoiceDate"].max()

    before_cutoff = before_cutoff.copy()
    before_cutoff["revenue"] = before_cutoff["Quantity"] * before_cutoff["Price"]

    features = pd.DataFrame()
    features["recency"] = (cutoff - last_purchase).dt.days
    features["tenure"] = (cutoff - first_purchase).dt.days
    features["frequency"] = before_cutoff.groupby("Customer ID")["Invoice"].nunique()
    features["monetary"] = before_cutoff.groupby("Customer ID")["revenue"].sum()

    features = features.reset_index()
    return features
    

In [139]:
features = build_features(data,CUTOFF)

In [153]:
def merge(lables,features):
    data = features.merge(lables,on = "Customer ID")
    return data

In [154]:
data = merge(labels,features)

In [155]:
data

,Customer ID,recency,tenure,frequency,monetary,returned
0,12346.0,233,633,17,-51.74,0
1,12347.0,37,312,6,3402.39,1
2,12348.0,156,346,4,1709.40,1
3,12349.0,315,643,4,2646.99,1
4,12350.0,218,218,1,334.40,0
...,...,...,...,...,...,...
5336,18283.0,3,566,16,1706.20,1
5337,18284.0,337,339,2,436.68,0
5338,18285.0,568,568,1,427.00,0
5339,18286.0,384,631,3,1188.43,0
